[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sharukh01621/first-line-to-first-paper/blob/main/colab_drive/solutions/part_04_data_science_math/ch19_visualization_storytelling/03_coding_exam_SOLUTION.ipynb)

> **From First Line to First Paper** — *Mastering Python, Statistical Data Science, and Applied Deep Learning*  
> **Author:** Mezanur Rahman Sharukh | **Mode:** `ANNOTATED SOLUTION NOTEBOOK`


# Chapter 19 Hands-On Coding Exam — Publication-Ready Figures & Pipelines (100 Points)

**Book Reference:** Chapter 19, Section 19.7 (Hands-On Coding Exam)  
**Suggested Time:** 40–50 minutes (Open Book, Closed Solution)

| Task | Publication Visualization Component | Points |
| :--- | :--- | :---: |
| **Task 19.1** | `build_mch_equity_panel(mch_df)` (1x2 stunting % & anemia % across wealth quintiles + `suptitle`) | 35 |
| **Task 19.2** | `build_outbreak_dashboard_figure(epi_df, district)` (2x1 shared-x dengue trend + rainfall bars) | 35 |
| **Task 19.3** | `export_figure_bundle(fig, out_dir, stem, dpi=300)` (Auto-mkdir + 300-DPI PNG + vector PDF bundle) | 30 |
| **TOTAL** | | **100** |

In [ ]:
# @title Step 0: Click Play (▶) to Activate Exam Auto-Grader
!pip install -q "git+https://github.com/sharukh01621/first-line-to-first-paper.git#subdirectory=sharukh_tutor"
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import sharukh_tutor as tutor

tutor.activate(chapter="ch19", notebook="coding_exam")


---
## Exam Task 19.1 (35 Points): `build_mch_equity_panel(mch_df)`

Build a 1x2 side-by-side health equity figure from `mch_df` (columns `wealth_quintile`, `child_stunted`, `maternal_anemia`):
1. Compute prevalence **percentages** (`mean() * 100`) grouped by `wealth_quintile` in logical order `["Poorest", "Poorer", "Middle", "Richer", "Richest"]`.
2. Create `fig, axes = plt.subplots(1, 2, figsize=(10.5, 4.2))`.
3. Left panel (`axes[0]`): bar chart of **stunting %** by wealth quintile with non-empty title, x-label, and y-label.
4. Right panel (`axes[1]`): bar chart of **maternal anemia %** by wealth quintile with non-empty title, x-label, and y-label.
5. Add an overall figure super-title via `fig.suptitle(...)` and return `(fig, axes)` without mutating `mch_df`.

In [ ]:
def build_mch_equity_panel(mch_df):
    """Build a 1x2 MCH stunting & anemia equity figure by wealth."""
    order = ["Poorest", "Poorer", "Middle", "Richer", "Richest"]
    cols = ["child_stunted", "maternal_anemia"]
    summary = (
        mch_df.groupby("wealth_quintile")[cols]
        .mean()
        .reindex(order)
        * 100
    )
    fig, axes = plt.subplots(1, 2, figsize=(10.5, 4.2))
    axes[0].bar(summary.index, summary["child_stunted"], color="#0072B2")
    axes[0].set_title("Child Stunting by Wealth Quintile")
    axes[0].set_xlabel("Wealth Quintile")
    axes[0].set_ylabel("Stunting Prevalence (%)")

    axes[1].bar(summary.index, summary["maternal_anemia"], color="#D55E00")
    axes[1].set_title("Maternal Anemia by Wealth Quintile")
    axes[1].set_xlabel("Wealth Quintile")
    axes[1].set_ylabel("Anemia Prevalence (%)")

    fig.suptitle("Maternal & Child Health Equity Across Wealth Quintiles")
    fig.tight_layout()
    return fig, axes


---
## Exam Task 19.2 (35 Points): `build_outbreak_dashboard_figure(epi_df, district)`

Build a 2x1 stacked outbreak dashboard for `district` from `epi_df` (columns `year`, `epi_week`, `district`, `dengue_cases`, `rainfall_mm`):
1. Filter `epi_df` to `district` and sort chronologically by `["year", "epi_week"]`.
2. Create `fig, axes = plt.subplots(2, 1, sharex=True, figsize=(8.0, 5.5))`.
3. Top panel (`axes[0]`): plot weekly `dengue_cases` + 4-week rolling mean line, add a legend, title, and y-label.
4. Bottom panel (`axes[1]`): plot weekly `rainfall_mm` as a bar chart (`axes[1].bar(...)`), with title, x-label, and y-label.
5. Return `(fig, axes)` without mutating `epi_df`.

In [ ]:
def build_outbreak_dashboard_figure(epi_df, district):
    """Build a 2x1 shared-x dengue incidence and rainfall dashboard."""
    sub = (
        epi_df[epi_df["district"] == district]
        .sort_values(["year", "epi_week"])
        .reset_index(drop=True)
    )
    x = range(1, len(sub) + 1)
    roll = sub["dengue_cases"].rolling(4, min_periods=1).mean()

    fig, axes = plt.subplots(2, 1, sharex=True, figsize=(8.0, 5.5))
    axes[0].plot(
        x, sub["dengue_cases"], color="#56B4E9", alpha=0.65,
        label="Weekly Cases",
    )
    axes[0].plot(
        x, roll, color="#D55E00", linewidth=2.2,
        label="4-Week Rolling Mean",
    )
    axes[0].set_title(f"Weekly Dengue Incidence — {district}")
    axes[0].set_ylabel("Dengue Cases")
    axes[0].legend(frameon=False)

    axes[1].bar(x, sub["rainfall_mm"], color="#0072B2", alpha=0.8)
    axes[1].set_title(f"Weekly Monsoon Rainfall — {district}")
    axes[1].set_xlabel("Sequential Epi Week")
    axes[1].set_ylabel("Rainfall (mm)")
    fig.tight_layout()
    return fig, axes


---
## Exam Task 19.3 (30 Points): `export_figure_bundle(fig, out_dir, stem, dpi=300)`

Build a reproducible camera-ready figure bundle exporter:
1. Ensure `out_dir` exists using `Path(out_dir).mkdir(parents=True, exist_ok=True)`.
2. Export both `<stem>.png` and `<stem>.pdf` inside `out_dir` at `dpi` with `bbox_inches="tight"`.
3. Return a dictionary with exact keys:
   - `"png"`: `Path` to `<stem>.png`
   - `"pdf"`: `Path` to `<stem>.pdf`
   - `"png_bytes"`: `int` file size of the PNG in bytes
   - `"pdf_bytes"`: `int` file size of the PDF in bytes

In [ ]:
def export_figure_bundle(fig, out_dir, stem, dpi=300):
    """Export tight PNG + PDF figure bundle and return path/byte map."""
    out = Path(out_dir)
    out.mkdir(parents=True, exist_ok=True)
    png_path = out / f"{stem}.png"
    pdf_path = out / f"{stem}.pdf"
    fig.savefig(png_path, dpi=dpi, bbox_inches="tight")
    fig.savefig(pdf_path, dpi=dpi, bbox_inches="tight")
    return {
        "png": png_path,
        "pdf": pdf_path,
        "png_bytes": int(png_path.stat().st_size),
        "pdf_bytes": int(pdf_path.stat().st_size),
    }


In [ ]:
exam_submissions = {
    "ch19_exam_01": build_mch_equity_panel,
    "ch19_exam_02": build_outbreak_dashboard_figure,
    "ch19_exam_03": export_figure_bundle,
}

tutor.check_exam("ch19", exam_submissions)


---
## Exam Debugging Post-Mortem (Solution Notebook Only)

- **Task 19.1 (Percentages vs. Raw Proportions & Quintile Ordering):** `groupby('wealth_quintile').mean()` sorts alphabetically (`Middle`, `Poorer`, `Poorest`, `Richer`, `Richest`) and yields `0..1` proportions. Always `.reindex(['Poorest', 'Poorer', 'Middle', 'Richer', 'Richest'])` and multiply by `100` so the socioeconomic gradient reads monotonically from left to right!
- **Task 19.2 (`sharex=True` & Avoiding Dual Y-Axes):** Overlaying dengue counts and rainfall millimeters on two different y-axes of the same panel distorts visual slopes depending on arbitrary axis scaling. Stacking two panels (`plt.subplots(2, 1, sharex=True)`) keeps both quantitative scales completely honest while aligning epidemiological weeks vertically.
- **Task 19.3 (Nested Output Directories & Dual Formats):** Calling `out.mkdir(parents=True, exist_ok=True)` prevents `FileNotFoundError` in fresh CI/manuscript builds, and exporting both 300-DPI PNG (raster preview) and vector PDF (lossless LaTeX/Quarto print) in one function eliminates figure drift.